In [1]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
from scipy import stats
from dash.dash_table.Format import Format, Scheme, Trim
# import ast
# import json
# import plotly.io as pio
# pio.renderers.default = "notebook_connected"

In [2]:
sigmas = [0.2, 0.5, 0.8, 1.1, 1.4, 1.7, 2.0]
reps = range(1)
suffixes = ['child_results', 'node_stats', 'parent_results', 'static_info', 'unary_results']

unary_list = []
child_list = []
parent_list = []
static_list = []


for sigma in sigmas:
    for rep in reps:
     
        # base_path = f"output/S{sigma}-R{rep}__"
        base_path = f"S{sigma}-R{rep}__"

        if os.path.exists(f"{base_path}unary_results.csv"):

            u_df = pd.read_csv(f"{base_path}unary_results.csv")
            u_df["sigma"] = sigma
            u_df["rep"] = rep
            unary_list.append(u_df)

            c_df = pd.read_csv(f"{base_path}child_results.csv")
            c_df["sigma"] = sigma
            c_df["rep"] = rep
            child_list.append(c_df)

            p_df = pd.read_csv(f"{base_path}parent_results.csv")
            p_df["sigma"] = sigma
            p_df["rep"] = rep
            parent_list.append(p_df)

            s_df = pd.read_csv(f"{base_path}static_info.csv")
            s_df["sigma"] = sigma
            s_df["rep"] = rep
            static_list.append(s_df)


master_unary_df = pd.concat(unary_list, ignore_index=True)
master_child_df = pd.concat(child_list, ignore_index=True)
master_parent_df = pd.concat(parent_list, ignore_index=True)
master_static_df = pd.concat(static_list, ignore_index=True)

In [3]:

def accuracy_gains(df, node_type="unary"):

    sts_col = f"{node_type}_sts_error"
    ets_col = f"{node_type}_ets_error"

    df["accuracy_gain"] = df[sts_col] - df[ets_col]

    unique_sigmas = sorted(df["sigma"].unique())
    num_plots = len(unique_sigmas)

    cols = min(3, num_plots)
    rows = (num_plots + cols - 1) // cols

    fig = make_subplots(
        rows=rows,
        cols=cols,
        shared_yaxes=False,
        horizontal_spacing=0.06,
        vertical_spacing=0.15,
    )

    for idx, sigma_val in enumerate(unique_sigmas):
        sigma_data = df[df["sigma"] == sigma_val]     
        
        gains = sigma_data["accuracy_gain"]
        sts_errors = sigma_data[sts_col]
        ets_errors = sigma_data[ets_col]
 
        r = (idx // cols) + 1
        c = (idx % cols) + 1

        t_stat, p_val = stats.ttest_ind(sts_errors, ets_errors)
        p_text = f"p = {p_val:.4e}"
        t_text = f"t = {t_stat:.4e}"
  

        fig.add_trace(
            go.Histogram(
                x=gains,
                name=f"sigma = {sigma_val}",
                nbinsx=50,
                marker=dict(
                    color="rgba(157, 174, 17, 0.9)",
                    line=dict(color="rgba(106, 118, 12, 1.0)", width=1),
                ),
                showlegend=False,
            ),
            row=r,
            col=c,
        )

        title_text = (
            f"<b>sigma = {sigma_val}</b>"
        )
        legend_text = (
            f"<b>{p_text}</b><br>" 
            f"<b>{t_text}</b><br>"
        )

        fig.add_annotation(
            text=title_text,
            xref=f"x{idx+1 if idx > 0 else ''} domain",
            yref=f"y{idx+1 if idx > 0 else ''} domain",
            x=0.5,
            y=1.1,
            showarrow=False,
            font=dict(size=12),
            align="center",
        )

        fig.add_annotation(
            text=legend_text,
            xref=f"x{idx+1 if idx > 0 else ''} domain",
            yref=f"y{idx+1 if idx > 0 else ''} domain",
            x=1,
            y=0.8,
            showarrow=False,
            font=dict(size=12),
            align="left",
        )



    if node_type == 'unary':
        title_type = 'Extended'
    elif node_type == 'parent':
        title_type = 'Parent'
    else:
        title_type = 'Child'




    fig.update_layout(
        title=dict(
            text=f"Significance of Accuracy Gains via Haplotype Extension in {title_type} Nodes",
            x=0.5,
            xanchor="center",
            font=dict(size=16),
        ),
        template="plotly_white",
        height=320 * rows + 120,
        width=360 * cols + 80,
        margin=dict(t=100, b=60, l=60, r=40),
    )
    
#  fig.update_xaxes(title_text="Bottom-Left X-Axis", row=2, col=1)



    fig.update_xaxes(title_text="Accuracy Difference (Simplified - Extended)", row=2, col=1)
    fig.update_xaxes(title_text="Accuracy Difference (Simplified - Extended)", row=2, col=2)
    fig.update_xaxes(title_text="Accuracy Difference (Simplified - Extended)", row=2, col=3)


    fig.update_yaxes(title_text="Frequency", row=1, col=1)
    fig.update_yaxes(title_text="Frequency", row=2, col=1)




    fig.show()
    return fig




In [4]:
def accuracy_vs_sigma_table(master_df, node_type):

    sts_col = f"{node_type}_sts_error"
    ets_col = f"{node_type}_ets_error"

    grouped = master_df.groupby(["sigma", "rep"])
    averaged = grouped[[sts_col, ets_col]].mean()
    rep_summary = averaged.reset_index()

    grouped_again = rep_summary.groupby("sigma")
    aggregated = grouped_again[[sts_col, ets_col]].agg(["mean"])
    stats_df = aggregated.reset_index()

 
    stats_df.columns = [
        "sigma",
        "sts_mean",
        "ets_mean",
    ]

    stats_df = stats_df.sort_values("sigma")

    best_index = ((stats_df["sts_mean"] - stats_df["ets_mean"]).abs()).idxmax()
 

    sigma_strings = []
    sts_strings = []
    ets_strings = []

    for i in stats_df.index:
        s_val = stats_df.loc[i, "sigma"]
        sts_val = stats_df.loc[i, "sts_mean"]
        ets_val = stats_df.loc[i, "ets_mean"]

        if i == best_index:
            sigma_strings.append(f"<b>{s_val:.1f}</b>")
            sts_strings.append(f"<b>{sts_val:.4f}</b>")
            ets_strings.append(f"<b>{ets_val:.4f}</b>")
        else:
            sigma_strings.append(f"{s_val:.1f}")
            sts_strings.append(f"{sts_val:.4f}")
            ets_strings.append(f"{ets_val:.4f}")
        
  


    fig = go.Figure(data=[go.Table(
        columnwidth=[50, 100, 100], 
        header=dict(values=['Sigma', 'Mean Simplified Error', 'Mean Extended Error'],
                    #  line_color='darkslategray',
                    fill_color='lightskyblue',
                    align='left'),
        cells=dict(values=[sigma_strings, sts_strings, ets_strings], 
            #    line_color='darkslategray',
                fill_color='lightcyan',
                align='left'
                # format=['.1f', '.4f', '.4f']
               ))

    ])

    fig.update_layout(width=650, height=700)
    fig.show()
    
    

In [5]:
def accuracy_vs_sigma(master_df, node_type):
    sts_col = f"{node_type}_sts_error"
    ets_col = f"{node_type}_ets_error"

    grouped = master_df.groupby(["sigma", "rep"])
    averaged = grouped[[sts_col, ets_col]].mean()
    rep_summary = averaged.reset_index()

    grouped_again = rep_summary.groupby("sigma")
    aggregated = grouped_again[[sts_col, ets_col]].agg(["mean", "sem"])
    stats_df = aggregated.reset_index()

 
    stats_df.columns = [
        "sigma",
        "sts_mean",
        "sts_sem",
        "ets_mean",
        "ets_sem",
    ]

    stats_df = stats_df.sort_values("sigma")
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=stats_df["sigma"],
            y=stats_df["sts_mean"],
            error_y=dict(
                type="data",
                array=stats_df["sts_sem"],
                visible=True,
                thickness=1.5,
                width=4,
            ),
            mode="lines+markers",
            name="Simplified",
            line=dict(color="rgba(253, 179, 056, 1.0)", width=2.5),
            marker=dict(size=8),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=stats_df["sigma"],
            y=stats_df["ets_mean"],
            error_y=dict(
                type="data",
                array=stats_df["ets_sem"],
                visible=True,
                thickness=1.5,
                width=4,
            ),
            mode="lines+markers",
            name="Extended",
            line=dict(color="rgba(002, 081, 150, 0.5)", width=2.5),
            marker=dict(size=8),
        )
    )

    if node_type == 'unary':
        title_type = 'Extended'
    elif node_type == 'parent':
        title_type = 'Parent'
    else:
        title_type = 'Child'

    fig.update_layout(
        title=dict(
            text=f"Total Accuracy as a Function of Spatial Scale (Sigma) in {title_type} Nodes",
            x=0.5,
            xanchor="center",
            font=dict(size=16),
        ),
        xaxis=dict(
            title="Spatial Dispersal (Sigma)",
            tickmode="array",
            tickvals=stats_df["sigma"],
        ),
        yaxis=dict(
            title="Log Mean Error", gridcolor="rgba(240, 240, 240, 1)"
        ),
        template="plotly_white",
        width=800,
        height=500,
        hovermode="x unified",
        legend=dict(yanchor="top", y=0.95, xanchor="right", x=0.95, bgcolor="rgba(255,255,255,0.8)"),
    )
    fig.update_yaxes(type='log')
    fig.show()
    


In [6]:

def accuracy_vs_time(master_df, node_type, num_bins):
  
    time_col = f"{node_type}_node_time"
    sts_col = f"{node_type}_sts_error"
    ets_col = f"{node_type}_ets_error"

    unique_sigmas = sorted(master_df["sigma"].unique())
    num_plots = len(unique_sigmas)


    cols = min(3, num_plots)
    rows = (num_plots + cols - 1) // cols

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"<b>sigma = {s}</b>" for s in unique_sigmas],
        horizontal_spacing=0.06,
        vertical_spacing=0.15,
        # shared_yaxes=True,
    )

    for idx, sigma_val in enumerate(unique_sigmas):
        sigma_data = master_df[master_df["sigma"] == sigma_val].copy()

        sigma_data["time_bin"] = pd.qcut(sigma_data[time_col], q=num_bins, labels=False, duplicates="drop")

        grouped_bin = sigma_data.groupby(["time_bin", "rep"])
        averaged_bin = grouped_bin[[time_col, sts_col, ets_col]].mean().reset_index()

        binned_df = (
            sigma_data.groupby("time_bin")
            .agg({time_col: "mean", sts_col: ["mean", "sem"], ets_col: ["mean", "sem"],}).reset_index())
        #     .sort_values(time_col)
        # )

        binned_df.columns = [
            "time_bin",
            "time_col",
            "sts_mean",
            "sts_sem",
            "ets_mean",
            "ets_sem",
        ]
        binned_df = binned_df.sort_values("time_col")

        r = (idx // cols) + 1
        c = (idx % cols) + 1


        fig.add_trace(
            go.Scatter(
                x=binned_df["time_col"],
                y=binned_df["sts_mean"],
                error_y=dict(
                    type="data",
                    array=binned_df["sts_sem"],
                    visible=True,
                    thickness=1.5,
                    width=4,
                ),
                mode="lines+markers",
                name="Simplified",
                line=dict(color="rgba(253, 179, 056, 1.0)", width=2),
                marker=dict(size=5),
                showlegend=(idx == 0),
            ),
            row=r,
            col=c,
        )
  
        fig.add_trace(
            go.Scatter(
                x=binned_df["time_col"],
                y=binned_df["ets_mean"],
                error_y=dict(
                    type="data",
                    array=binned_df["ets_sem"],
                    visible=True,
                    thickness=1.5,
                    width=4,
                ),
                mode="lines+markers",
                name="Extended",
                line=dict(color="rgba(002, 081, 150, 1)", width=2),
                marker=dict(size=5),
                showlegend=(idx == 0),
            ),
            row=r,
            col=c,
        )

    if node_type == 'unary':
        title_type = 'Extended'
    elif node_type == 'parent':
        title_type = 'Parent'
    else:
        title_type = 'Child'
   
    fig.update_layout(
        title=dict(
            text=f"Error Across Time in {title_type} Nodes",
            x=0.5,
            xanchor="center",
            font=dict(size=16),
        ),
        template="plotly_white",
        height=320 * rows + 120,
        width=360 * cols + 80,
        margin=dict(t=140, b=60, l=60, r=40),
        hovermode="x unified",
    )

    fig.update_yaxes(matches='y')
    fig.update_xaxes(title_text="Node Time")
    fig.update_yaxes(title_text="Mean Error")
    fig.show()
   

In [7]:
def all_in_one(master_df, node_type, num_bins):

    time_col = f"{node_type}_node_time"
    sts_col = f"{node_type}_sts_error"
    ets_col = f"{node_type}_ets_error"

    unique_sigmas = sorted(master_df["sigma"].unique())
    fig = go.Figure()
    color_list = {"rgba(194, 106, 119, 1.0)", 
                  "rgba(046, 037, 133, 1.0)",
                  "rgba(051, 117, 056, 1.0)",
                  "rgba(093, 168, 153, 1.0)",
                  "rgba(148, 203, 236, 1.0)",
                  "rgba(220, 205, 125, 1.0)"}
    
    for idx, (sigma_val, coolor) in enumerate(zip(unique_sigmas, color_list)):
        sigma_data = master_df[master_df["sigma"] == sigma_val].copy()

        sigma_data["time_bin"] = pd.qcut(sigma_data[time_col], q=num_bins, labels=False, duplicates="drop")

        binned_df = (
            sigma_data.groupby("time_bin")
            .agg({time_col: "mean", sts_col: "mean", ets_col: "mean"})
            .sort_values(time_col)
        )

        fig.add_trace(
            go.Scatter(
                x=binned_df[time_col],
                y=binned_df[sts_col],
                mode="lines+markers",
                name=f"{sigma_val} Simplified",
                line=dict(color=coolor, width=2, dash='dash'),
                marker=dict(size=5),
            ),
        )

        fig.add_trace(
            go.Scatter(
                x=binned_df[time_col],
                y=binned_df[ets_col],
                mode="lines+markers",
                name=f"{sigma_val} Extended",
                line=dict(color=coolor, width=2),
                marker=dict(size=5),
            ),
        )

    if node_type == 'unary':
        title_type = 'Extended'
    elif node_type == 'parent':
        title_type = 'Parent'
    else:
        title_type = 'Child'
   
    fig.update_layout(
        title=dict(
            text=f"Error Across Time in {title_type} Nodes",
            x=0.5,
            xanchor="center",
            font=dict(size=16),
        ),
        template="plotly_white",
        margin=dict(t=140, b=60, l=60, r=40),
        hovermode="x unified",
    )

    fig.update_xaxes(title_text="Node Time")
    fig.update_yaxes(title_text="Mean Error")
    # fig.update_yaxes(type='log')
    fig.show()
    return

In [8]:
def accuracy_vs_total_extension(master_df, node_type, num_bins):
  
    span_col = "added_span"
    sts_col = f"{node_type}_sts_error"
    ets_col = f"{node_type}_ets_error"

    unique_sigmas = sorted(master_df["sigma"].unique())
    num_plots = len(unique_sigmas)


    cols = min(3, num_plots)
    rows = (num_plots + cols - 1) // cols

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"<b>sigma = {s}</b>" for s in unique_sigmas],
        horizontal_spacing=0.06,
        vertical_spacing=0.15,
    )

    for idx, sigma_val in enumerate(unique_sigmas):
        sigma_data = master_df[master_df["sigma"] == sigma_val].copy()

        sigma_data["span_bin"] = pd.qcut(sigma_data[span_col], q=num_bins, labels=False, duplicates="drop")

        binned_df = (
            sigma_data.groupby("span_bin")
            .agg({span_col: "mean", sts_col: ["mean", "sem"], ets_col: ["mean", "sem"]}).reset_index())
        #     .sort_values(span_col)
        # )

        r = (idx // cols) + 1
        c = (idx % cols) + 1

        binned_df.columns = [
            "span_bin",
            "span_col",
            "sts_mean",
            "sts_sem",
            "ets_mean",
            "ets_sem",
        ]
        binned_df = binned_df.sort_values("span_col")

        fig.add_trace(
            go.Scatter(
                x=binned_df["span_col"],
                y=binned_df["sts_mean"],
                error_y=dict(
                    type="data",
                    array=binned_df["sts_sem"],
                    visible=True,
                    thickness=1.5,
                    width=4,
                ),
                mode="lines+markers",
                name="Simplified",
                line=dict(color="rgba(253, 179, 056, 1.0)", width=2),
                marker=dict(size=5),
                showlegend=(idx == 0),
            ),
            row=r,
            col=c,
        )

  
        fig.add_trace(
            go.Scatter(
                x=binned_df["span_col"],
                y=binned_df["ets_mean"],
                error_y=dict(
                    type="data",
                    array=binned_df["ets_sem"],
                    visible=True,
                    thickness=1.5,
                    width=4,
                ),
                mode="lines+markers",
                name="Extended",
                line=dict(color="rgba(002, 081, 150, 1)", width=2),
                marker=dict(size=5),
                showlegend=(idx == 0),
            ),
            row=r,
            col=c,
        )

    if node_type == 'unary':
        title_type = 'Extended'
    elif node_type == 'parent':
        title_type = 'Parent'
    else:
        title_type = 'Child'
   
    fig.update_layout(
        title=dict(
            text=f"Error Across Total Extension in {title_type} Nodes",
            x=0.5,
            xanchor="center",
            font=dict(size=16),
        ),
        template="plotly_white",
        height=320 * rows + 120,
        width=360 * cols + 80,
        margin=dict(t=140, b=60, l=60, r=40),
        hovermode="x unified",
    )

    fig.update_yaxes(matches='y')
    fig.update_xaxes(title_text="Amount Extended")
    fig.update_yaxes(title_text="Mean Error")
 
    
    fig.show()
   

In [9]:
def accuracy_vs_correct_extension(master_df, node_type, num_bins):
  
    span_col = "added_span"
    wrong_col = "wrongly_added_span"
    sts_col = f"{node_type}_sts_error"
    ets_col = f"{node_type}_ets_error"
    correct_span_col = "correct_span"

    unique_sigmas = sorted(master_df["sigma"].unique())
    num_plots = len(unique_sigmas)

    


    cols = min(3, num_plots)
    rows = (num_plots + cols - 1) // cols

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"<b>sigma = {s}</b>" for s in unique_sigmas],
        horizontal_spacing=0.06,
        vertical_spacing=0.15,
    )

    for idx, sigma_val in enumerate(unique_sigmas):
        sigma_data = master_df[master_df["sigma"] == sigma_val].copy()
        
        sigma_data[correct_span_col] = sigma_data[span_col] - sigma_data[wrong_col]

        sigma_data["span_bin"] = pd.qcut(sigma_data[correct_span_col], q=num_bins, labels=False, duplicates="drop")

        binned_df = (
            sigma_data.groupby("span_bin")
            .agg({correct_span_col: "mean", span_col: "mean", sts_col: "mean", ets_col: "mean"})
            .sort_values(correct_span_col)
        )

        r = (idx // cols) + 1
        c = (idx % cols) + 1


        fig.add_trace(
            go.Scatter(
                x=binned_df[correct_span_col],
                y=binned_df[sts_col],
                mode="lines+markers",
                name="Simplified",
                line=dict(color="rgba(253, 179, 056, 1.0)", width=2),
                marker=dict(size=5),
                showlegend=(idx == 0),
            ),
            row=r,
            col=c,
        )

  
        fig.add_trace(
            go.Scatter(
                x=binned_df[correct_span_col],
                y=binned_df[ets_col],
                mode="lines+markers",
                name="Extended",
                line=dict(color="rgba(002, 081, 150, 1)", width=2),
                marker=dict(size=5),
                showlegend=(idx == 0),
            ),
            row=r,
            col=c,
        )

    if node_type == 'unary':
        title_type = 'Extended'
    elif node_type == 'parent':
        title_type = 'Parent'
    else:
        title_type = 'Child'
   
    fig.update_layout(
        title=dict(
            text=f"Error Across Correct Extension in {title_type} Nodes",
            x=0.5,
            xanchor="center",
            font=dict(size=16),
        ),
        template="plotly_white",
        height=320 * rows + 120,
        width=360 * cols + 80,
        margin=dict(t=140, b=60, l=60, r=40),
        hovermode="x unified",
    )

    fig.update_yaxes(matches='y')
    fig.update_xaxes(title_text="Amount Extended Correctly")
    fig.update_yaxes(title_text="Mean Error")

    
    fig.show()
   

In [28]:
def accuracy_vs_proportion_extension(master_df, node_type, num_bins):
  
    span_col = "added_span"
    wrong_col = "wrongly_added_span"
    sts_col = f"{node_type}_sts_error"
    ets_col = f"{node_type}_ets_error"
    prop_col = "prop_correct_span"

    unique_sigmas = sorted(master_df["sigma"].unique())
    num_plots = len(unique_sigmas)

    cols = min(3, num_plots)
    rows = (num_plots + cols - 1) // cols

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"<b>sigma = {s}</b>" for s in unique_sigmas],
        horizontal_spacing=0.06,
        vertical_spacing=0.15,
    )

    for idx, sigma_val in enumerate(unique_sigmas):
        sigma_data = master_df[master_df["sigma"] == sigma_val].copy()

        correct_span = sigma_data[span_col] - sigma_data[wrong_col]

        sigma_data[prop_col] = (correct_span / sigma_data[span_col]).fillna(0)

        sigma_data["span_bin"] = pd.qcut(sigma_data[prop_col], q=num_bins, labels=False, duplicates="drop")
        
        
        binned_df = (
            sigma_data.groupby("span_bin")
            .agg({prop_col: "mean", span_col: "mean", sts_col: "mean", ets_col: "mean"})
            .sort_values(prop_col)
        )

        r = (idx // cols) + 1
        c = (idx % cols) + 1


        fig.add_trace(
            go.Scatter(
                x=binned_df[prop_col],
                y=binned_df[sts_col],
                mode="lines+markers",
                name="Simplified",
                line=dict(color="rgba(253, 179, 056, 1.0)", width=2),
                marker=dict(size=5),
                showlegend=(idx == 0),
            ),
            row=r,
            col=c,
        )

  
        fig.add_trace(
            go.Scatter(
                x=binned_df[prop_col],
                y=binned_df[ets_col],
                mode="lines+markers",
                name="Extended",
                line=dict(color="rgba(002, 081, 150, 1)", width=2),
                marker=dict(size=5),
                showlegend=(idx == 0),
            ),
            row=r,
            col=c,
        )

    if node_type == 'unary':
        title_type = 'Extended'
    elif node_type == 'parent':
        title_type = 'Parent'
    else:
        title_type = 'Child'
   
    fig.update_layout(
        title=dict(
            text=f"Error Across the Proportion of Correct Extension in {title_type} Nodes",
            x=0.5,
            xanchor="center",
            font=dict(size=16),
        ),
        template="plotly_white",
        height=320 * rows + 120,
        width=360 * cols + 80,
        margin=dict(t=140, b=60, l=60, r=40),
        hovermode="x unified",
    )

    fig.update_yaxes(matches='y')
    fig.update_xaxes(title_text="Proportion of Correct Extension")
    fig.update_yaxes(title_text="Mean Error")

    
    fig.show()
   

In [29]:
accuracy_vs_proportion_extension(master_unary_df, "unary", 200)

In [14]:
accuracy_vs_correct_extension(master_unary_df, "unary", 25)

In [16]:
accuracy_vs_total_extension(master_unary_df, "unary", 25)

In [18]:
all_in_one(master_unary_df, "unary", 200)

In [19]:
accuracy_vs_time(master_unary_df, "unary", 50)

In [20]:
accuracy_vs_sigma_table(master_unary_df, "unary")

In [21]:

accuracy_vs_sigma(master_unary_df, "unary")

In [22]:
accuracy_gains(master_unary_df, node_type="unary")
